In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import dask.dataframe as dd  # For scalability
import logging
from joblib import dump
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors
import json

/Users/phionanamugga/Documents/coding/datascience/EDA_Viral_Social_Media_Posts/.venv/lib/python3.9/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# Simulated Azure Blob Storage Data Loading
def load_data_from_azure(blob_name):
    logging.info(f"Simulating data load from Azure Blob Storage: {blob_name}")
    df = dd.read_csv("Viral_Social_Media_Trends.csv").compute()  
    return df

In [5]:
# Data Pipeline with Business Features
def process_data(df):
    logging.info("Processing data in pipeline")
    df.dropna(inplace=True)
    for col in ['Platform', 'Hashtag', 'Content_Type', 'Region', 'Engagement_Level']:
        df[col] = df[col].astype('category')
    numeric_cols = ['Views', 'Likes', 'Shares', 'Comments']
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    df['Engagement_Score'] = df[numeric_cols].mean(axis=1)
    df['Log_Views'] = np.log1p(df['Views'].clip(lower=0))
    # Business feature: Engagement-to-Views Ratio
    df['Engagement_to_Views'] = df['Engagement_Score'] / (df['Views'] + 1e-6)  # Avoid division by zero
    return df, scaler

In [6]:
# NLP Features
def add_nlp_features(df):
    logging.info("Adding NLP features")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['Hashtag'].tolist(), show_progress_bar=True)
    kmeans = KMeans(n_clusters=3, random_state=42)
    df['Hashtag_Cluster'] = kmeans.fit_predict(embeddings)
    cluster_names = {0: 'Trendy', 1: 'Informative', 2: 'Casual'}
    df['Cluster_Name'] = df['Hashtag_Cluster'].map(cluster_names)
    
    # Sentiment analysis (BERT as placeholder, Azure Cognitive Services could be used)
    from transformers import pipeline
    sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    df['Hashtag_Sentiment'] = df['Hashtag'].apply(
        lambda x: sentiment_analyzer(x)[0]['score'] if sentiment_analyzer(x)[0]['label'] == 'POSITIVE' else -sentiment_analyzer(x)[0]['score']
    )
    
    top_hashtags = df.groupby('Platform').apply(lambda x: x.loc[x['Engagement_Score'].idxmax(), 'Hashtag']).to_dict()
    df['Top_Hashtag_Platform'] = df['Platform'].map(top_hashtags)
    return df

In [7]:
# Predictive Modeling with Hyperparameter Tuning
def train_predictive_model(df):
    logging.info("Training XGBoost model with hyperparameter tuning")
    features = ['Views', 'Likes', 'Shares', 'Comments', 'Hashtag_Sentiment', 'Hashtag_Cluster', 'Engagement_to_Views']
    X = df[features]
    y = df['Engagement_Score']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Hyperparameter tuning with GridSearchCV
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.3]
    }
    xgb_model = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
    grid_search = GridSearchCV(xgb_model, param_grid, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    logging.info(f"Best Parameters: {grid_search.best_params_}")
    logging.info(f"Model Performance - MSE: {mse:.4f}, R2: {r2:.4f}")
    
    # Simulate Azure ML deployment
    dump(best_model, 'xgb_model_azure.joblib')
    logging.info("Model saved as 'xgb_model_azure.joblib' (simulating Azure ML deployment)")
    return best_model, mse, r2